# 01 · Recall labels and EDA

Looks at how complaints relate to recalls and sanity-checks `data/processed/recall_labels_stg.parquet`
(built with `python -m src.recall_model.labels --source stg`).

Keys: make × model × model year × component, read through `src/recall_model/sources.py`.
`SOURCE = "stg"` uses the cleaned `stg.complaints` / `stg.recalls` tables; set it to `"raw"` to compare.
Complaints are dated by `ldate`, recalls by `recall_date`, vehicles only. Complaints cover 2010 to 2026.

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.recall_model.sources import get_sql

SOURCE = "stg"
complaints_sql, recalls_sql = get_sql(SOURCE)
con = duckdb.connect(str(ROOT / "data" / "nhtsa.duckdb"), read_only=True)
LABELS = (ROOT / "data" / "processed" / f"recall_labels_{SOURCE}.parquet").as_posix()

con.sql(f"create or replace temp table c as select *, date_trunc('month', ldate)::date as month from ({complaints_sql})")
con.sql(f"create or replace temp table r as select * from ({recalls_sql})")
con.sql(f"create or replace temp view lab as select * from read_parquet('{LABELS}')")
pd.set_option("display.width", 140)

## 1 · The label panel

In [ ]:
con.sql("""
select count(*) as cell_months, count(distinct (make, model, model_year, component)) as cells,
       round(100*avg(y_3m::int), 3) as pct_y_3m, round(100*avg(y_6m::int), 3) as pct_y_6m, round(100*avg(y_12m::int), 3) as pct_y_12m,
       round(100*avg(recalled_prev_12m::int), 2) as pct_recalled_prev_12m
from lab""").df()

In [ ]:
by_year = con.sql("""
select year(month) as yr, count(*) as cell_months, round(100*avg(y_6m::int), 3) as pct_y_6m,
       sum(n_complaints) as complaints
from lab group by 1 order by 1""").df()
by_year

## 2 · Can complaints see these recalls at all?

For vehicle recalls received 2016–2019 (so there is at least a year of complaint history), how many
recalled cells had any complaint on the same make/model/year/component *before* the recall date?

In [ ]:
cov = con.sql("""
with rc as (
  select make, model, model_year, component, min(recall_date) as recall_date, any_value(camp_no) as camp_no
  from r where recall_date between date '2016-01-01' and date '2019-12-31' group by all
),
pre as (
  select rc.*,
    (select count(*) from c where c.make=rc.make and c.model=rc.model and c.model_year=rc.model_year
       and c.component=rc.component and c.ldate < rc.recall_date) as n_before,
    (select count(*) from c where c.make=rc.make and c.model=rc.model and c.model_year=rc.model_year
       and c.component=rc.component and c.ldate < rc.recall_date
       and c.ldate >= rc.recall_date - interval 12 month) as n_12m_before,
    exists(select 1 from c where c.make=rc.make and c.model=rc.model and c.model_year=rc.model_year) as vehicle_has_complaints
  from rc
)
select * from pre""").df()

summary = pd.Series({
    "recalled cells 2016-19": len(cov),
    "vehicle has any complaints": cov.vehicle_has_complaints.mean(),
    ">=1 complaint on same component before recall": (cov.n_before >= 1).mean(),
    ">=5 in 12 months before recall": (cov.n_12m_before >= 5).mean(),
    ">=1, among vehicles with complaints": (cov.n_before[cov.vehicle_has_complaints] >= 1).mean(),
})
summary.to_frame("value")

## 3 · How do complaints move around a recall? (event study)

Monthly complaints on recalled cells, aligned on the recall month (month 0), averaged over recalled cells
that had at least one complaint beforehand. If the curve rises well before month 0, there is warning to be had.

In [ ]:
ev = con.sql("""
with rc as (
  select make, model, model_year, component, date_trunc('month', min(recall_date))::date as rmonth
  from r where recall_date between date '2017-01-01' and date '2019-06-30' group by all
),
cm as (select make, model, model_year, component, month, count(*) as n from c group by all),
grid as (select rc.*, k from rc, range(-24, 7) t(k)),
j as (
  select g.*, coalesce(cm.n, 0) as n
  from grid g left join cm on cm.make=g.make and cm.model=g.model and cm.model_year=g.model_year
    and cm.component=g.component and cm.month = g.rmonth + to_months(g.k)
),
keep as (select make, model, model_year, component from j where k < 0 group by all having sum(n) > 0)
select k as months_from_recall, avg(n) as mean_complaints, median(n) as median_complaints, count(*) as cells
from j semi join keep using (make, model, model_year, component)
group by k order by k""").df()

base = con.sql("select avg(n_complaints) from lab where not y_12m and not recalled_prev_12m").fetchall()[0][0]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(ev.months_from_recall, ev.mean_complaints, marker="o", ms=3, label="recalled cells (mean)")
ax.axhline(base, color="grey", ls="--", lw=1, label=f"non-recalled cell-months (mean {base:.2f})")
ax.axvline(0, color="k", lw=1)
ax.set_xlabel("months from recall (RCDATE)"); ax.set_ylabel("complaints / month"); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f"Complaints around a recall ({int(ev.cells.iloc[0]):,} recalled cells with prior complaints)")
plt.show()
ev.set_index("months_from_recall").T.round(2)

## 4 · Lead time: first complaint and first "surge" before the recall

For recalled cells with prior complaints: months from the first complaint (since 2010) to the recall, and
months from the first month with ≥3 complaints. This bounds how many weeks of warning are even possible.

In [ ]:
lead = con.sql("""
with rc as (
  select make, model, model_year, component, min(recall_date) as recall_date
  from r where recall_date between date '2016-01-01' and date '2019-12-31' group by all
),
cm as (select make, model, model_year, component, month, count(*) as n from c group by all),
j as (select rc.*, cm.month, cm.n from rc join cm using (make, model, model_year, component) where cm.month < date_trunc('month', rc.recall_date))
select make, model, model_year, component, recall_date,
       datediff('month', min(month), recall_date) as months_first_complaint,
       datediff('month', min(month) filter (where n >= 3), recall_date) as months_first_surge,
       sum(n) as complaints_before
from j group by make, model, model_year, component, recall_date""").df()
lead[["months_first_complaint", "months_first_surge", "complaints_before"]].describe(percentiles=[.25, .5, .75, .9]).round(1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
lead.months_first_complaint.plot.hist(bins=48, ax=ax[0], title="months: first complaint → recall")
lead.months_first_surge.dropna().plot.hist(bins=48, ax=ax[1], title="months: first month with ≥3 complaints → recall")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5 · Who starts the recall? (manufacturer vs. NHTSA)

In [ ]:
con.sql("""
select INFLUENCED_BY, count(distinct CAMPNO) as campaigns
from raw.recalls where RCLTYPECD='V' and RCDATE between '20160101' and '20191231'
group by 1 order by 2 desc""").df()

## 6 · Components: volume and recall rate

In [ ]:
con.sql("""
select component, count(*) as cell_months, sum(n_complaints) as complaints,
       sum(y_6m::int) as pos_6m, round(100*avg(y_6m::int), 2) as pct_y_6m
from lab group by 1 order by complaints desc limit 20""").df()

## 7 · Does complaint volume separate positives? (quick check, not a model)

In [ ]:
con.sql("""
with f as (
  select *, sum(n_complaints) over (partition by make, model, model_year, component order by month
             rows between 2 preceding and current row) as n_3m
  from lab where not recalled_prev_12m
)
select case when n_3m = 0 then '0' when n_3m < 3 then '1-2' when n_3m < 10 then '3-9'
            when n_3m < 30 then '10-29' else '30+' end as complaints_last_3m,
       count(*) as cell_months, round(100*avg(y_6m::int), 2) as pct_y_6m
from f group by 1 order by min(n_3m)""").df()

## Takeaways

Numbers are from the cleaned tables (`SOURCE = "stg"`), complaints 2010 to Oct 2026, run on Nov 16.

- **Panel:** 16.0M cell-months over 136k cells. Positive rate is 0.29% at 3 months, 0.53% at 6 and 0.94% at 12, so use PR-AUC / precision@K, not accuracy.
- **Most recalls are invisible to complaints.** Of 22.4k recalled cells (2016–19), 43% are on a vehicle with any complaint and 13% had a same-component complaint before the recall (30% among vehicles with complaints). On raw 2015–19 data this was 9%, so the cleaned names and the extra years help but the gap is still large. Report coverage on recalls that had prior complaints, as well as overall.
- **There is a signal.** On recalled cells with prior complaints, mean complaints/month go from about 0.4 two years before the recall to about 0.75 just before it, versus 0.08 for non-recalled cell-months. The median is 0, so it comes from a minority of cells.
- **Lead time is long.** Median 32 months from first complaint to recall (25th percentile 13 months). With only 2015–19 complaints loaded the median looked like 16 months, because earlier complaints were cut off. A 6-month horizon is reasonable; 12 months is worth trying.
- **Volume alone helps a lot.** The 6-month positive rate goes from 0.29% with no complaints in the last 3 months to 7.3% with 30+.
- **Air bags are the biggest part but no longer most of it:** 21.2k of 85.1k positive cell-months (25%). Still check results with and without air bags.
- **The component map fixed fuel.** On raw names fuel had 0 positives; with the cleaned names `FUEL/PROPULSION SYSTEM` has 5.8k. `UNKNOWN OR OTHER` has 106k complaints and almost no recalls (0.04%).
- Manufacturers start most campaigns (3,300 of 3,520 in 2016–19); NHTSA (ODI) starts 207.